---
# COSC2673 | COSC2793 (Computational) Machine Learning
## Week 3 Lab: **Dataset Splitting & Pre-Processing**
---

# Introduction

Last week, we learned how to read data and do exploratory data analysis (EDA). The next step in a typical machine learning pipeline is to split data and transform it so that we can feed the data to a learning algorithm.

The lab assumes that you have completed ``Week 02 Lab: Reading Data & Exploratory Data Analysis (EDA)``. If you haven't yet, please do so before attempting this lab.

## Objective
- Continue to familiarise with Python and machine learning packages.
- Learn to split the data to training, (validation) and test sets.
- Important considerations in splitting the data.
- Learn to transform (pre‑process) data: data encoding and normalization. 


## Dataset

We will examine two regression datasets. The first concerns house prices and factors related to them;
the second is daily bike-share rentals in Washington, D.C., USA, with features for time and weather.
The datasets are `housing.data.csv` and `bikeShareDay.csv` in the repository.

First, ensure both data files are available in your Jupyter workspace. Copy the data directories (`BostonHousingPrice` and `Bike-Sharing-Dataset`) into the current working folder.

# Data Splitting

As we have discussed in the lecture, in supervised learning we are interested in learning a model using our dataset that can predict the target value for unseen data (outside the training set). This is called **generalization**. How can we test if the model we developed with our training data would generalize? One approach is to **hold some data back from the training process** (hypothetical unseen data). This held‑out subset is called the **test set** and the remaining data is the **training set**. The training set may be further subdivided into a **training set** and **validation set** to select and tune the hyper-parameters. We can use the test set at the end of the development phase to evaluate our model's ability to generalize.

- **Training set:** Used to train or fit your model. For example, you use the training set to find the optimal weights or coefficients for linear regression, logistic regression, or neural networks.
- **Test set:** Needed for an unbiased evaluation of the final model.

The test set should be independent and identically distributed (i.i.d) with respect to the training data.

Make sure there is no leakage between the two sets (no overlapping instances). Leakage gives unrealistically high performance; e.g. in house price prediction a single property sold multiple times could appear in both train and test.

There should be no underlying differences between the distributions. For example, if the train set contains houses sold only in winter and the test set only in summer, the characteristics could differ.

Do not use test data for any part of model development (training). This includes hyper-parameter tuning and model selection; use a separate validation set for those tasks.

## Random splitting

In machine learning, the most common approach is random splitting to create train and test splits. We randomly select instances for the training set and assign the remainder to the test set. The key configuration is the split ratio, typically specified as a fraction for the training or test set. For example, a training set size of 0.67 (67%) means the remaining 0.33 (33%) is used for testing.

There is no universally optimal split percentage.

Choose a split that meets your project's objectives with considerations including:

- Computational cost for training/evaluation.
- Representativeness of training/test sets.

Common choices are:

- Train: 80 %, Test: 20 %
- Train: 67 %, Test: 33 %
- Train: 50 %, Test: 50 %

Let's first load the house price dataset.

Use the knowledge from last week to load the dataset into a pandas DataFrame named `bostonHouseFrame`.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
# plt.style.use('seaborn-v0_8')

## TODO
bostonHouseFrame = pd.read_csv('./Bos')

The scikit-learn Python machine learning library provides an implementation of the train-test splitting via function `train_test_split()`. Lets use this to randomly split our data to 80% training and 20% testing.

In [2]:
from sklearn.model_selection import train_test_split

# split data randomly into 80% train and 20% test (fixed seed for reproducibility)
bostonHouseTrainFrame, bostonHouseTestFrame = train_test_split(
    bostonHouseFrame, test_size=0.2, shuffle=True, random_state=42
)

NameError: name 'bostonHouseFrame' is not defined

In [ ]:
print(
    f"Number of instances in the original dataset: {bostonHouseFrame.shape[0]}.\n",
    f"After splitting, train set has {bostonHouseTrainFrame.shape[0]} instances",
    f"and test set has {bostonHouseTestFrame.shape[0]} instances.",
)  # use f-string for clarity

## Checking your splits

As discussed, random splitting may lead to leakage (two splits are not independent). We need to understand the dataset to make sure there are no hidden sources of leakage in the data. This is one place we can use the knowledge we gained through EDA.

Use the EDA observations from last week to see if there is any issue with the random splitting process.

We can use histogram plots to see if the two partitions (splits) are identically distributed. 

Use the knowledge from last week to plot the histograms for each attribute for the two splits. Use different colors for training vs test.

What observations did you make?

**Observations:** 

- The distribution of training set attributes is approximately equal to the distribution of test set attributes. 


Make sure you use the same bins for both plots (train/test).

Now you know how to randomly split the data. Random splitting is not the only method to split data. The method used may vary based on many factors like problem type, nature of data etc. For example if we have time series data, we can use [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html). 

It is also common in machine learning to write your own custom function to do data splitting where special measures are required to keep the data independent or identical. 

Write your own function to split the dataset into training, validation and test sets, then use EDA to validate your implementation or even better implement some unit tests.

In [ ]:
def train_validation_test_split(dateset, validation_size, test_size, shuffle, random_state):
    ## TODO
    # return training_set, validation_set, test_set
    pass

# Data Pre-processing (or Transforming)

The data you read from a CSV file or dataset may not be of the form that is suitable for machine learning algorithms. Therefore it is common to perform some preprocessing of the data. Usually data preprocessing involves several distinct steps:

1. Cleaning data & Removing/filling missing values.
2. Encoding data
3. Feature scaling

We will focus on the the latter two for now. 


## Feature Scaling

In a typical dataset, you might have different numerical features with widely different ranges. For example, during EDA, we discovered that the attribute NOX ranges `[0,1]` whereas TAX takes ranges `[0, 700]`. Furthermore, you may have some features that have a [skewed distribution](https://www.statisticshowto.com/probability-and-statistics/skewed-distribution/). Such characteristics in data may cause problems for learning algorithms (specially gradient based methods and distance based methods). Therefore it is common to use feature scaling.

Important: Feature scaling, is usually guided by the EDA. The histograms and other individual feature visualizations often provided useful information for feature scaling and can be used to justify one approach over another. 

Two most common methods employed for feature normalization are min-max scaling and standard scaling:

 - **Min-max scaling:** An individual feature is transformed so that the values are mapped to the range `[0,1]`. [Ref](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html#sklearn.preprocessing.MinMaxScaler).
 - **Standard scaling:** An individual feature is transformed so that the transformed values have zero mean and unit variance. [Ref](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler).

Lets apply the above two methods to the feature `RM` in the Boston house price dataset.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

MinMaxScaler_RM = MinMaxScaler().fit(bostonHouseTrainFrame[['RM']])
RM_minmax = MinMaxScaler_RM.transform(bostonHouseTrainFrame[['RM']])

StandardScaler_RM = StandardScaler().fit(bostonHouseTrainFrame[['RM']])
RM_standard = StandardScaler_RM.transform(bostonHouseTrainFrame[['RM']])

Now lets plot the feature distribution before and after to see the difference

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.hist(bostonHouseTrainFrame['RM'], alpha=0.3, color='r', density=True)
plt.title("Original")

plt.subplot(1,3,2)
plt.hist(RM_minmax, alpha=0.3, color='r')
plt.title("After Min-Max scaling")

plt.subplot(1,3,3)
plt.hist(RM_standard, alpha=0.3, color='r')
plt.title("After Standard Scaling")
plt.tight_layout()
plt.show()

What observations did you make?

**Observations:** 
- Both scaling methods do not change the shape of the feature distribution. They only change the range. 

Let's go with standard scaling as it tends to be more robust to outliers.

In [ ]:
# Apply the transformation to train data and save in the dataframe
bostonHouseTrainFrame['RM'] = StandardScaler_RM.transform(bostonHouseTrainFrame[['RM']])

# Apply the transformation to test data and save in the dataframe
bostonHouseTestFrame['RM'] = StandardScaler_RM.transform(bostonHouseTestFrame[['RM']])

Warning: When normalizing, ensure that the same scaling parameters are applied to all splits (train/validation/test).

A common mistake is to use one set of scaling parameters to normalize the training data and another for the test data. This happens if you apply `fit_transform()` function twice: once to the training set and again for the test data.

The correct approach would be to fit() on the training data and then apply the transform() to the training set and test set separately, to scale the data. 

Now implement standard scaling yourself without using sklearn then validate your implementation by comparing your results to those obtained from sklearn.

In [ ]:
## TODO

We can also use non-linear transformation(s) to map a feature that has a skewed distribution to have a distribution that is close to a Gaussian which facilitates learning for some machine learning algorithms. 

Task: Read this article on [skewed distribution](https://www.statisticshowto.com/probability-and-statistics/skewed-distribution/). 

Lets try a non-linear transformation with attribute `DIS`.

In [ ]:
from sklearn.preprocessing import PowerTransformer

# transform DIS (skewed) with Yeo-Johnson power transform
power_transformer_DIS = PowerTransformer(method='yeo-johnson', standardize=False).fit(bostonHouseTrainFrame[['DIS']])
DIS_power = power_transformer_DIS.transform(bostonHouseTrainFrame[['DIS']])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.hist(bostonHouseTrainFrame['DIS'], alpha=0.3, color='r', density=True)
plt.title("Original")

plt.subplot(1,2,2)
plt.hist(DIS_power, alpha=0.3, color='r')
plt.title("Power scaling")
plt.tight_layout()
plt.show()

In [ ]:
bostonHouseTrainFrame['DIS'] = power_transformer_DIS.transform(bostonHouseTrainFrame[['DIS']])
bostonHouseTestFrame['DIS'] = power_transformer_DIS.transform(bostonHouseTestFrame[['DIS']])

Task: Select the appropriate feature scaling method for all numerical attributes in the Boston house dataset.

There are many other normalization techniques you can try. See [scikit-learn preprocessing documentation](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing) for more information. 

There are also methods that can handle outliers. See [Compare the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html#sphx-glr-auto-examples-preprocessing-plot-all-scaling-py).

## Encoding Categorical Data

So far we looked at how to pre-process numerical data. What about categorical data? As we discussed last week, categorical data can come in two forms:

Categorical Variables: These are data points that take on a finite number of values, AND whose values do not have a numerical interpretation.

- Ordinal categorical variables take on values which can be logically ordered. For example, the reviews for a product which are given as 0-5 stars. 

- Nominal categorical variables cannot be put in any logical order. Examples of this would be colours etc.


There is only one categorical attribute in the Boston house price dataset (`CHAS`). This attribute is already pre-processed and stored as a binary variable (0,1). Therefore no further preprocessing is required. To learn about the encoding process, we will now switch to the bike share dataset. 

Load the bike share dataset and examine the attributes. 

In [ ]:
## TODO


What are the `ordinal categorical variables` and `nominal categorical variables`?

For ordinal categorical variables, each category can be represented as an integer. This process is called label encoding. Label encoding is simply converting each value in a column to a number. 

Lets take the attribute `weathersit` which explains the weather situation. This attribute can take four values:

- 1: Clear, Few clouds, Partly cloudy, Partly cloudy
- 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
- 3: Light snow, Light rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
- 4: Heavy rain + Ice pallets + Thunderstorm + Mist, Snow + Fog

*Note: The dataset does not contain rows with value 4.*

This is a variable that cannot be put in any logical order (one might argue it can be, but for this lab, let's go with the above assumption). You can represent categorical values as numbers, but you won't be able to compare these numbers or subtract them from each other. For example, the value of 1 is obviously less than the value of 4 but does that really correspond to the dataset in real life? Does a heavy rain has “4X” more weight in our calculation than the clear?

What can we do to rectify this issue?

A common approach is called **one hot encoding**. The basic strategy is to convert each category value into a new column and assign a 1 or 0 (True/False) value to the column. This has the benefit of not weighting a value improperly but does have the downside of adding more columns to the dataset.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

OneHotEncoder_weathersit = OneHotEncoder(handle_unknown='ignore')
OneHotEncoder_weathersit.fit(bikeShareFrame[['weathersit']])

onehot_ = OneHotEncoder_weathersit.transform(bikeShareFrame[['weathersit']]).toarray()

print(onehot_.shape, OneHotEncoder_weathersit.categories_)

We can see that the `weathersit` column is now converted to 3 columns. The first column will say if the day is "Clear, Few clouds, Partly cloudy, Partly cloudy" or not, and so on. 

Now we can put these columns back to the dataframe.

In [ ]:
colName = 'weathersit'
for i in range(len(OneHotEncoder_weathersit.categories_[0])):
    bikeShareFrame[colName + '_' + str(OneHotEncoder_weathersit.categories_[0][i])] = onehot_[:,i]

    
bikeShareFrame.head()

Now we have three extra columns. compare them with the original 'weathersit' column and see if it matches.

Since we have represented the 'weathersit' column with the three new columns, we should remove the original column to eliminate redundancy.

In [ ]:
bikeShareFrame = bikeShareFrame.drop(['weathersit'], axis=1)
bikeShareFrame.head()

There are even more advanced algorithms for categorical encoding. [This](https://www.kdnuggets.com/2015/12/beyond-one-hot-exploration-categorical-variables.html) article provides some additional technical background and comparisons. A good explanation of the other methods are given in [here](https://stats.idre.ucla.edu/r/library/r-library-contrast-coding-systems-for-categorical-variables/#backward), however the code is in R.  

# Exercise: Split and Pre-process the Bike Share Data

Task: Do the splitting and numerical feature scaling on the Bike Share Data.</font>

Now that you have seen how to do this task for the house price dataset. Repeat the same process for the Daily
Bike Share rental dataset.

Answer the following questions and discuss this with your tutor. Please do attempt this, and don't wait to see if solutions are released (they will not be!)
- How to avoid leakage in the dataset?
- What type of normalization should be used for each attribute?